email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [162]:
import os
from dotenv import load_dotenv

load_dotenv()


True

In [163]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [164]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [165]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [166]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")

    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

In [167]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [168]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model=os.environ.get("OPENAI_MODEL"),
    temperature=0.3,
)

agent = create_agent(
    model=model,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="My email address is julie@example.com and the pwd is password123. Send a mail with the content draft 1 to zinyi073@gmail.com")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

d:\workspace\coding\langchian-academy\lca-lc-foundations\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


In [170]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

draft 1


In [171]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

The email with the content "draft 1" has been successfully sent to zinyi073@gmail.com.


In [172]:
from rich import print as rprint

rprint(response)

{
    'messages': [
        HumanMessage(
            content='My email address is julie@example.com and the pwd is password123. Send a mail with the content
draft 1 to zinyi073@gmail.com',
            additional_kwargs={},
            response_metadata={},
            id='a974739a-9c8c-4ba7-85b1-906ebc7502e6'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 42,
                    'prompt_tokens': 330,
                    'total_tokens': 372,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'Qwen3.5-397B-A17B-FP8',
                'system_fingerprint': None,
                'id': 'chatcmpl-bf167ddf31ba1586',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019d32e0-cd5d-76d2-a5a4-ab63b07f63a4-0',
            tool_calls=[
                {
                    'name': 'authenticate',
                    'args': {'email': 'julie@example.com', 'password': 'password123'},
                    'id': 'chatcmpl-tool-b73fae3e2c7f3401',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 330,
                'output_tokens': 42,
                'total_tokens': 372,
                'input_token_details': {},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='Successfully authenticated',
            name='authenticate',
            id='64fb5442-308c-4edd-80db-513d16000546',
            tool_call_id='chatcmpl-tool-b73fae3e2c7f3401'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 62,
                    'prompt_tokens': 446,
                    'total_tokens': 508,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'Qwen3.5-397B-A17B-FP8',
                'system_fingerprint': None,
                'id': 'chatcmpl-a256585b890911b2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019d32e0-cf00-7662-b55f-dd21456df8ad-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'to': 'zinyi073@gmail.com', 'subject': 'draft 1', 'body': 'draft 1'},
                    'id': 'chatcmpl-tool-b2914608e48cd173',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 446,
                'output_tokens': 62,
                'total_tokens': 508,
                'input_token_details': {},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='Email sent to zinyi073@gmail.com with subject draft 1 and body draft 1',
            name='send_email',
            id='c716eaa5-84bf-4a24-8109-88980ce12f8f',
            tool_call_id='chatcmpl-tool-b2914608e48cd173'
        ),
        AIMessage(
            content='The email with the content "draft 1" has been successfully sent to zinyi073@gmail.com.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 25,
                    'prompt_tokens': 546,
                    'total_tokens': 571,
                    'completion_tokens_details': None,
                    'prompt_tok

In [173]:
from langchain_core.messages import messages_to_dict

result = messages_to_dict(response["messages"])

contents = [m["data"]["content"] for m in result]

rprint(contents)

[
    'My email address is julie@example.com and the pwd is password123. Send a mail with the content draft 1 to 
zinyi073@gmail.com',
    '',
    'Successfully authenticated',
    '',
    'Email sent to zinyi073@gmail.com with subject draft 1 and body draft 1',
    'The email with the content "draft 1" has been successfully sent to zinyi073@gmail.com.'
]